# ELEN4025 – Machine Learning Group Project
## Stage 1: Data Loading & Sanity Checks

In [2]:
#Set Up
import pandas as pd
import numpy as np
import os
import subprocess
import zipfile

RAW_DIR = os.path.join("data", "raw")
PROCESSED_DIR = os.path.join("data", "processed")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

EXPECTED_TABLES = {
    "studentInfo":         {"min_rows": 30000, "key_cols": ["code_module", "code_presentation", "id_student"]},
    "studentVle":          {"min_rows": 10000000, "key_cols": ["code_module", "code_presentation", "id_student", "id_site", "date"]},
    "assessments":         {"min_rows": 200, "key_cols": ["code_module", "code_presentation", "id_assessment"]},
    "studentAssessment":   {"min_rows": 170000, "key_cols": ["id_assessment", "id_student"]},
    "studentRegistration": {"min_rows": 30000, "key_cols": ["code_module", "code_presentation", "id_student"]},
    "courses":             {"min_rows": 20, "key_cols": ["code_module", "code_presentation"]},
    "vle":                 {"min_rows": 6000, "key_cols": ["id_site", "code_module", "code_presentation"]},
}

print("Configuration ready.")
print(f"  RAW_DIR:       {RAW_DIR}")
print(f"  PROCESSED_DIR: {PROCESSED_DIR}")
print(f"  Expected tables: {list(EXPECTED_TABLES.keys())}")

Configuration ready.
  RAW_DIR:       data/raw
  PROCESSED_DIR: data/processed
  Expected tables: ['studentInfo', 'studentVle', 'assessments', 'studentAssessment', 'studentRegistration', 'courses', 'vle']


In [3]:
#Download and Extract Dataset 
ZIP_PATH = os.path.join("data", "oulad.zip")

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DIR)
        extracted = zf.namelist()
    print(f"Extracted {len(extracted)} files from {ZIP_PATH}:")
    for f in sorted(extracted):
        print(f"  {f}")
else:
    print(f"Zip file not found at {ZIP_PATH}.")
    print("Checking if CSVs already exist in data/raw/ ...")
    existing = [f for f in os.listdir(RAW_DIR) if f.endswith(".csv")]
    print(f"  Found {len(existing)} CSV files: {existing}")

Extracted 7 files from data/oulad.zip:
  assessments.csv
  courses.csv
  studentAssessment.csv
  studentInfo.csv
  studentRegistration.csv
  studentVle.csv
  vle.csv


In [4]:
#Check all CSV files have been exraccted
expected_files = [f"{name}.csv" for name in EXPECTED_TABLES]
for fname in expected_files:
    path = os.path.join(RAW_DIR, fname)
    assert os.path.exists(path), f"FAIL: Missing file {path}"
    size_mb = os.path.getsize(path) / 1e6
    print(f"   {fname:30s} ({size_mb:.1f} MB)")

print(f"\n All {len(expected_files)} CSV files present.")

   studentInfo.csv                (3.5 MB)
   studentVle.csv                 (453.8 MB)
   assessments.csv                (0.0 MB)
   studentAssessment.csv          (5.7 MB)
   studentRegistration.csv        (1.1 MB)
   courses.csv                    (0.0 MB)
   vle.csv                        (0.3 MB)

 All 7 CSV files present.


In [5]:
# Load CSV Tables 
tables = {}
for name in EXPECTED_TABLES:
    path = os.path.join(RAW_DIR, f"{name}.csv")
    tables[name] = pd.read_csv(path)
    print(f"Loaded {name:25s} -> {tables[name].shape[0]:>10,} rows x {tables[name].shape[1]:>3} cols")

print(f"\nAll {len(tables)} tables loaded.")

Loaded studentInfo               ->     32,593 rows x  12 cols
Loaded studentVle                -> 10,655,280 rows x   6 cols
Loaded assessments               ->        206 rows x   6 cols
Loaded studentAssessment         ->    173,912 rows x   5 cols
Loaded studentRegistration       ->     32,593 rows x   5 cols
Loaded courses                   ->         22 rows x   3 cols
Loaded vle                       ->      6,364 rows x   6 cols

All 7 tables loaded.


In [6]:
#Check and confirm all tables have been loaded correctly 
assert len(tables) == 7, f"FAIL: Expected 7 tables, got {len(tables)}"
print("All 7 tables loaded into memory.")

for name, df in tables.items():
    assert len(df) > 0, f"FAIL: {name} is empty"
print("No empty tables.")

for name, df in tables.items():
    assert df.shape[0] >= EXPECTED_TABLES[name]["min_rows"], (
        f"FAIL: {name} has {df.shape[0]} rows, expected >= {EXPECTED_TABLES[name]['min_rows']}"
    )
print("All row counts in expected range.")



All 7 tables loaded into memory.
No empty tables.
All row counts in expected range.


In [7]:
#Inspect Data Types and Shapes
for name, df in tables.items():
    print(f"\n{'─'*55}")
    print(f"  {name}  |  {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"{'─'*55}")
    print(df.dtypes.to_string())
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())


───────────────────────────────────────────────────────
  studentInfo  |  32,593 rows x 12 cols
───────────────────────────────────────────────────────
code_module               str
code_presentation         str
id_student              int64
gender                    str
region                    str
highest_education         str
imd_band                  str
age_band                  str
num_of_prev_attempts    int64
studied_credits         int64
disability                str
final_result              str

First 3 rows:
  code_module code_presentation  id_student gender                region      highest_education imd_band age_band  num_of_prev_attempts  studied_credits disability final_result
0         AAA             2013J       11391      M   East Anglian Region       HE Qualification  90-100%     55<=                     0              240          N         Pass
1         AAA             2013J       28400      F              Scotland       HE Qualification   20-30%    35-55     

In [8]:
#Verify Key Columns Exist with correct types 
for name, df in tables.items():
    for col in EXPECTED_TABLES[name]["key_cols"]:
        assert col in df.columns, f"FAIL: {name} missing column '{col}'"
print("All key columns present in every table.")

# Check specific expected column counts
assert tables["studentInfo"].shape[1] == 12, "FAIL: studentInfo should have 12 columns"
assert tables["studentVle"].shape[1] == 6, "FAIL: studentVle should have 6 columns"
assert tables["courses"].shape[1] == 3, "FAIL: courses should have 3 columns"
assert tables["assessments"].shape[1] == 6, "FAIL: assessments should have 6 columns"
assert tables["studentAssessment"].shape[1] == 5, "FAIL: studentAssessment should have 5 columns"
assert tables["studentRegistration"].shape[1] == 5, "FAIL: studentRegistration should have 5 columns"
assert tables["vle"].shape[1] == 6, "FAIL: vle should have 6 columns"
print(" All Column counts correct for all tables.")

# sum_click must be numeric for aggregation; id_student must be numeric for joins.
# If pandas read them as strings, downstream computations would silently fail.
assert tables["studentVle"]["sum_click"].dtype in [np.int64, np.int32], "FAIL: sum_click should be integer"
assert tables["studentInfo"]["id_student"].dtype in [np.int64, np.int32], "FAIL: id_student should be integer"
print("Critical columns have expected dtypes.")



All key columns present in every table.
 All Column counts correct for all tables.
Critical columns have expected dtypes.


## Missing/ Null Vlue Analysis 

In [9]:
#Identify Missing Data 
print("Missing value analysis per table:\n")

missing_summary = []
for name, df in tables.items():
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"  {name}: ✓ no missing values")
    else:
        miss = df.isnull().sum()
        miss = miss[miss > 0]
        for col, count in miss.items():
            pct = 100.0 * count / len(df)
            missing_summary.append({
                "Table": name,
                "Column": col,
                "Missing Count": count,
                "Missing %": round(pct, 2),
            })
            print(f"  {name}.{col}: {count:,} missing ({pct:.2f}%)")

Missing value analysis per table:

  studentInfo.imd_band: 1,111 missing (3.41%)
  studentVle: ✓ no missing values
  assessments.date: 11 missing (5.34%)
  studentAssessment.score: 173 missing (0.10%)
  studentRegistration.date_registration: 45 missing (0.14%)
  studentRegistration.date_unregistration: 22,521 missing (69.10%)
  courses: ✓ no missing values
  vle.week_from: 5,243 missing (82.39%)
  vle.week_to: 5,243 missing (82.39%)


In [10]:
#Table shwoing missing values summary
missing_df = pd.DataFrame(missing_summary)
print("\n─── Missing Values Summary Table ───\n")
print(missing_df.to_string(index=False))


─── Missing Values Summary Table ───

              Table              Column  Missing Count  Missing %
        studentInfo            imd_band           1111       3.41
        assessments                date             11       5.34
  studentAssessment               score            173       0.10
studentRegistration   date_registration             45       0.14
studentRegistration date_unregistration          22521      69.10
                vle           week_from           5243      82.39
                vle             week_to           5243      82.39


In [11]:
#Document Missing Values and Confirm Critical Columns are Clean
si = tables["studentInfo"]

for col in ["id_student", "final_result", "code_module", "code_presentation", "gender", "region"]:
    assert si[col].isnull().sum() == 0, f"FAIL: studentInfo.{col} has nulls"
print("No missing values in critical studentInfo columns.")


assert tables["studentVle"].isnull().sum().sum() == 0, "FAIL: studentVle has nulls"
print("studentVle has zero missing values.")


assert tables["courses"].isnull().sum().sum() == 0, "FAIL: courses has nulls"
print("courses has zero missing values.")


assert si["imd_band"].isnull().sum() == 1111, "FAIL: imd_band missing count unexpected"
print("imd_band: 1,111 missing.")

unreg_missing = tables["studentRegistration"]["date_unregistration"].isnull().sum()
assert unreg_missing == 22521, f"FAIL: date_unregistration missing count unexpected: {unreg_missing}"
print(f"date_unregistration: {unreg_missing:,} missing.")

score_missing = tables["studentAssessment"]["score"].isnull().sum()
assert score_missing == 173, f"FAIL: score missing count unexpected: {score_missing}"
print(f"score: {score_missing} missing.")

print("\n All missing values documented.")

No missing values in critical studentInfo columns.
studentVle has zero missing values.
courses has zero missing values.
imd_band: 1,111 missing.
date_unregistration: 22,521 missing.
score: 173 missing.

 All missing values documented.


In [12]:
#Identify and remove duplicate rows 
print("Duplicate analysis per table:\n")
print(f"{'Table':25s} {'Duplicates':>12s}   {'Decision':30s}")
print("─" * 75)

for name, df in tables.items():
    n_dup = df.duplicated().sum()
    if name == "studentVle" and n_dup > 0:
        decision = "KEEP"
    elif n_dup > 0:
        decision = "REMOVE"
    else:
        decision = "OK — none found"
    print(f"{name:25s} {n_dup:>12,}   {decision}")


Duplicate analysis per table:

Table                       Duplicates   Decision                      
───────────────────────────────────────────────────────────────────────────
studentInfo                          0   OK — none found
studentVle                     787,170   KEEP
assessments                          0   OK — none found
studentAssessment                    0   OK — none found
studentRegistration                  0   OK — none found
courses                              0   OK — none found
vle                                  0   OK — none found


### Duplicate Handling

`studentVle` duplicates are kept because they represent valid repeated clicks on the same resource on the same day. Removing them would undercount real VLE engagement.

All other tables have zero duplicates, so no duplicate removal was required.

In [13]:
#Raw Distrabution of final Results
si = tables["studentInfo"].copy()

print("Raw final_result distribution:\n")
counts = si["final_result"].value_counts()
for val, count in counts.items():
    pct = 100 * count / len(si)
    print(f"  {val:15s}: {count:>6,}  ({pct:.1f}%)")
print(f"\n  Total registrations: {len(si):,}")


Raw final_result distribution:

  Pass           : 12,361  (37.9%)
  Withdrawn      : 10,156  (31.2%)
  Fail           :  7,052  (21.6%)
  Distinction    :  3,024  (9.3%)

  Total registrations: 32,593


In [14]:
# Note: "Withdrawn" is the second-largest group (31.2%).
# These are students who de-registered before completing the course.
# Treating them as "unfavourable" is appropriate: from an early-warning
# perspective, we want to flag students at risk of EITHER failing or dropping out.

### Step 6b — Map to binary target

The target is **unfavourable vs favourable outcome**:

| Original `final_result` | Binary `target` | Category |
|---|---|---|
| Pass | 1 | Favourable |
| Distinction | 1 | Favourable |
| Fail | 0 | Unfavourable |
| Withdrawn | 0 | Unfavourable |

**Design decision:** We group `Withdrawn` with `Fail` because the university's
goal is early intervention: identifying students who might either fail or
drop out so that support can be offered. A student who withdraws is just as
much a "lost" outcome as one who fails.

In [15]:
LABEL_MAP = {
    "Pass": 1,         # Favourable — completed and passed
    "Distinction": 1,  # Favourable — completed with distinction
    "Fail": 0,         # Unfavourable — completed but did not pass
    "Withdrawn": 0,    # Unfavourable — dropped out before completion
}

si["target"] = si["final_result"].map(LABEL_MAP)

target_counts = si["target"].value_counts().sort_index()
print("Binary target distribution:\n")
for label, count in target_counts.items():
    pct = 100 * count / len(si)
    tag = "favourable (Pass/Distinction)" if label == 1 else "unfavourable (Fail/Withdrawn)"
    print(f"  target = {label} — {tag}: {count:>6,}  ({pct:.1f}%)")

class_ratio = target_counts[1] / target_counts[0]
print(f"\nClass ratio (favourable / unfavourable): {class_ratio:.3f}")
print(f"Imbalance: {'Moderate — no resampling strictly needed' if 0.5 < class_ratio < 2.0 else 'Significant — consider resampling'}")

tables["studentInfo"] = si

Binary target distribution:

  target = 0 — unfavourable (Fail/Withdrawn): 17,208  (52.8%)
  target = 1 — favourable (Pass/Distinction): 15,385  (47.2%)

Class ratio (favourable / unfavourable): 0.894
Imbalance: Moderate — no resampling strictly needed


In [16]:
#Check Binary Target is valid
# No unmapped values — if final_result contained a category we didn't anticipate,
# map() would produce NaN, which would silently corrupt all model training.
assert si["target"].isnull().sum() == 0, "FAIL: Some final_result values were not mapped!"
print("All final_result values successfully mapped (no nulls in target).")

# Only {0, 1} — multi-class labels would break the binary classifiers
assert set(si["target"].unique()) == {0, 1}, "FAIL: Target is not binary {0, 1}!"
print("Target is strictly binary: {0, 1}.")

# Verify each mapping individually to catch accidental swaps
assert (si.loc[si["final_result"] == "Pass", "target"] == 1).all(), "FAIL: Pass != 1"
assert (si.loc[si["final_result"] == "Distinction", "target"] == 1).all(), "FAIL: Distinction != 1"
assert (si.loc[si["final_result"] == "Fail", "target"] == 0).all(), "FAIL: Fail != 0"
assert (si.loc[si["final_result"] == "Withdrawn", "target"] == 0).all(), "FAIL: Withdrawn != 0"
print("Mapping verified: Pass→1, Distinction→1, Fail→0, Withdrawn→0.")

# Row count must be unchanged — mapping should not add or drop any rows
assert len(si) == 32593, f"FAIL: Expected 32,593 rows, got {len(si)}"
print("Row count unchanged: 32,593.")

# Class balance check — if either class drops below 10%, standard metrics
# like accuracy become misleading and we'd need to rely on F1/AUC exclusively.
min_pct = 100 * target_counts.min() / len(si)
assert min_pct > 10, f"FAIL: Minority class only {min_pct:.1f}%"
print(f"Minority class = {min_pct:.1f}% — moderate imbalance, workable with stratified splits.")

# counts must add up to total rows
assert target_counts.sum() == len(si), "FAIL: Target counts don't sum to total rows"
print(f"Target counts sum to total: {target_counts.sum():,}.")

print("\n Binary target created and validated.")

All final_result values successfully mapped (no nulls in target).
Target is strictly binary: {0, 1}.
Mapping verified: Pass→1, Distinction→1, Fail→0, Withdrawn→0.
Row count unchanged: 32,593.
Minority class = 47.2% — moderate imbalance, workable with stratified splits.
Target counts sum to total: 32,593.

 Binary target created and validated.


In [17]:
#Referential Integrity Checks
# The composite key (code_module, code_presentation, id_student) uniquely
# identifies each student-course registration. All child tables must reference
# only keys that exist in studentInfo.
si_keys = set(zip(si["code_module"], si["code_presentation"], si["id_student"]))

# Check 1: Every student in the click-log must exist in studentInfo,
# otherwise their engagement features can't be linked to an outcome.
svle = tables["studentVle"]
svle_keys = set(zip(svle["code_module"], svle["code_presentation"], svle["id_student"]))
orphan_vle = svle_keys - si_keys

# Check 2: Registration records must match. mismatches would indicate
# students who registered but have no demographic/outcome data.
sreg = tables["studentRegistration"]
sreg_keys = set(zip(sreg["code_module"], sreg["code_presentation"], sreg["id_student"]))
orphan_reg = sreg_keys - si_keys

# Check 3: Assessment IDs in student submissions must reference valid assessments.
# Invalid IDs would mean scores can't be tied to their assessment type/weight/date.
assess_ids = set(tables["assessments"]["id_assessment"])
sa_assess_ids = set(tables["studentAssessment"]["id_assessment"])
orphan_sa_ids = sa_assess_ids - assess_ids

# Check 4: Students who submitted assessments must exist in studentInfo.
# We need to join through assessments first to get the module/presentation context.
sa_merged = tables["studentAssessment"].merge(
    tables["assessments"][["id_assessment", "code_module", "code_presentation"]],
    on="id_assessment", how="left"
)
sa_keys = set(zip(sa_merged["code_module"], sa_merged["code_presentation"], sa_merged["id_student"]))
orphan_sa_stu = sa_keys - si_keys

print("Referential Integrity Checks:\n")
print(f"  studentVle students NOT in studentInfo:            {len(orphan_vle)}")
print(f"  studentRegistration students NOT in studentInfo:   {len(orphan_reg)}")
print(f"  studentAssessment assessment IDs NOT in assessments: {len(orphan_sa_ids)}")
print(f"  studentAssessment students NOT in studentInfo:     {len(orphan_sa_stu)}")

assert len(orphan_vle) == 0
assert len(orphan_reg) == 0
assert len(orphan_sa_ids) == 0
assert len(orphan_sa_stu) == 0
print("\n All join keys are consistent.")




Referential Integrity Checks:

  studentVle students NOT in studentInfo:            0
  studentRegistration students NOT in studentInfo:   0
  studentAssessment assessment IDs NOT in assessments: 0
  studentAssessment students NOT in studentInfo:     0

 All join keys are consistent.


In [18]:
#Full summary Table 
summary_rows = []
for name, df in tables.items():
    summary_rows.append({
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Cells": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })

summary_table = pd.DataFrame(summary_rows)
print("─── Sanity-Check Summary Table ───\n")
print(summary_table.to_string(index=False))

─── Sanity-Check Summary Table ───

              Table     Rows  Columns  Missing Cells  Duplicate Rows  Memory (MB)
        studentInfo    32593       13           1111               0         17.3
         studentVle 10655280        6              0          787170       1470.4
        assessments      206        6             11               0          0.0
  studentAssessment   173912        5            173               0          7.0
studentRegistration    32593        5          22566               0          4.2
            courses       22        3              0               0          0.0
                vle     6364        6          10486               0          1.2


In [19]:
# Save processed output locally
# Note: data/processed/ is ignored by Git, so this file is regenerated when the notebook is run.
out_path = os.path.join(PROCESSED_DIR, "studentInfo_with_target.csv")

si.to_csv(out_path, index=False)

print(f"Saved locally: {out_path}")
print(f"{len(si):,} rows, {si.shape[1]} columns")

Saved locally: data/processed/studentInfo_with_target.csv
32,593 rows, 13 columns


In [20]:
# Verify saved output
si_check = pd.read_csv(out_path)

assert si_check.shape == si.shape, f"FAIL: Saved shape {si_check.shape} != {si.shape}"
assert "target" in si_check.columns, "FAIL: 'target' column missing"
assert si_check["target"].isnull().sum() == 0, "FAIL: target has nulls"
assert set(si_check["target"].unique()) == {0, 1}, "FAIL: target not {0,1}"

saved_counts = si_check["target"].value_counts().sort_index()
assert (saved_counts == target_counts).all(), "FAIL: target distribution changed"

print("Output file verified.")
print(f"Distribution preserved: 0 = {saved_counts[0]:,}, 1 = {saved_counts[1]:,}")

Output file verified.
Distribution preserved: 0 = 17,208, 1 = 15,385


## Stage 3: Feature Engineering

In [21]:
WEEK_CUTOFFS = [2, 4, 6, 8]
DAYS_PER_WEEK = 7

KEYS = ["code_module", "code_presentation", "id_student"]


# Load Stage 1 outputs + Load VLE activity

In [22]:
def load_stage3_inputs():
    student_info_path = os.path.join(PROCESSED_DIR, "studentInfo_with_target.csv")

    if os.path.exists(student_info_path):
        student_info = pd.read_csv(student_info_path)
    else:
        raise FileNotFoundError(
            "Missing data/processed/studentInfo_with_target.csv. "
            "Run Stage 1 first so that the binary target is available."
        )

    student_vle = pd.read_csv(os.path.join(RAW_DIR, "studentVle.csv"))
    vle = pd.read_csv(os.path.join(RAW_DIR, "vle.csv"))

    # Load assessment data
    # Used for cutoff-safe performance features
    assessments = pd.read_csv(os.path.join(RAW_DIR, "assessments.csv"))
    student_assessment = pd.read_csv(os.path.join(RAW_DIR, "studentAssessment.csv"))

    return student_info, student_vle, vle, assessments, student_assessment

# Merge VLE metadata

In [23]:
def add_activity_type(student_vle, vle):
    vle_cols = [
        "id_site",
        "code_module",
        "code_presentation",
        "activity_type"
    ]

    return student_vle.merge(
        vle[vle_cols],
        on=["id_site", "code_module", "code_presentation"],
        how="left"
    )


# Merge Assessment Data

In [24]:
def add_assessment_metadata(student_assessment, assessments):
    assessment_cols = [
        "id_assessment",
        "code_module",
        "code_presentation",
        "assessment_type",
        "date",
        "weight"
    ]

    return student_assessment.merge(
        assessments[assessment_cols],
        on="id_assessment",
        how="left"
    )

# Aggregate VLE features

In [25]:
def aggregate_vle_until_cutoff(student_vle_enriched, cutoff_week):

    # Filter by cutoff week
    # Prevents future-data leakage
    cutoff_day = cutoff_week * DAYS_PER_WEEK

    vle_cut = student_vle_enriched[
        student_vle_enriched["date"] <= cutoff_day
    ].copy()


    # Aggregate total engagement
    # One row per student-course

    base_agg = vle_cut.groupby(KEYS).agg(
        total_clicks=("sum_click", "sum"),
        mean_daily_clicks=("sum_click", "mean"),
        max_daily_clicks=("sum_click", "max"),
        active_days=("date", "nunique"),
        unique_sites=("id_site", "nunique"),
        unique_activity_types=("activity_type", "nunique")
    ).reset_index()

    # Aggregate by activity type
    # Creates forum/resource/quiz click features
    activity_clicks = (
        vle_cut
        .pivot_table(
            index=KEYS,
            columns="activity_type",
            values="sum_click",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    activity_clicks.columns = [
        col if col in KEYS else f"clicks_{str(col).lower().replace(' ', '_')}"
        for col in activity_clicks.columns
    ]

    # Combine VLE feature groups
    # Joins total and activity-type features

    features = base_agg.merge(activity_clicks, on=KEYS, how="left")

    # Create ratio features
    # Normalised engagement measures

    features["clicks_per_active_day"] = (
        features["total_clicks"] / features["active_days"].replace(0, np.nan)
    ).fillna(0)

    features["clicks_per_site"] = (
        features["total_clicks"] / features["unique_sites"].replace(0, np.nan)
    ).fillna(0)

    features["cutoff_week"] = cutoff_week
    features["cutoff_day"] = cutoff_day

    return features

# Aggregate Assessment Features

In [26]:
def aggregate_assessments_until_cutoff(assessment_enriched, cutoff_week):

    # Filter by cutoff week
    # Prevents future assessment leakage
    cutoff_day = cutoff_week * DAYS_PER_WEEK

    assess_cut = assessment_enriched[
        assessment_enriched["date_submitted"] <= cutoff_day
    ].copy()


    # Remove banked assessments
    # Keeps only real submissions in current presentation

    if "is_banked" in assess_cut.columns:
        assess_cut = assess_cut[assess_cut["is_banked"] == 0].copy()

   
    # Weighted score contribution
    # Captures assessment importance

    assess_cut["weighted_score"] = (
        assess_cut["score"] * assess_cut["weight"] / 100
    )

    # Aggregate assessment performance
    # One row per student-course
   
    assessment_features = assess_cut.groupby(KEYS).agg(
        num_assessments_submitted=("id_assessment", "nunique"),
        mean_assessment_score=("score", "mean"),
        max_assessment_score=("score", "max"),
        min_assessment_score=("score", "min"),
        total_weighted_score=("weighted_score", "sum"),
        mean_days_submitted=("date_submitted", "mean"),
        earliest_submission_day=("date_submitted", "min"),
        latest_submission_day=("date_submitted", "max")
    ).reset_index()

 
    # Assessment-type counts
    # Counts submitted TMAs, CMAs, Exams, etc.

    assessment_type_counts = (
        assess_cut
        .pivot_table(
            index=KEYS,
            columns="assessment_type",
            values="id_assessment",
            aggfunc="nunique",
            fill_value=0
        )
        .reset_index()
    )

    assessment_type_counts.columns = [
        col if col in KEYS else f"num_{str(col).lower()}_submitted"
        for col in assessment_type_counts.columns
    ]

  
    # Assessment-type mean scores
    # Captures performance by assessment category

    assessment_type_scores = (
        assess_cut
        .pivot_table(
            index=KEYS,
            columns="assessment_type",
            values="score",
            aggfunc="mean",
            fill_value=0
        )
        .reset_index()
    )

    assessment_type_scores.columns = [
        col if col in KEYS else f"mean_{str(col).lower()}_score"
        for col in assessment_type_scores.columns
    ]

    # Combine assessment feature groups
    # Joins totals, counts, and type scores

    assessment_features = assessment_features.merge(
        assessment_type_counts,
        on=KEYS,
        how="left"
    )

    assessment_features = assessment_features.merge(
        assessment_type_scores,
        on=KEYS,
        how="left"
    )

    return assessment_features

# Merge with student demographics

In [27]:
def build_week_feature_table(student_info, vle_features, assessment_features, cutoff_week):
    demographic_cols = [
        "code_module",
        "code_presentation",
        "id_student",
        "gender",
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability",
        "final_result",
        "target"
    ]

    df = student_info[demographic_cols].merge(
        vle_features,
        on=KEYS,
        how="left"
    )

    # ------------------------------------------------------------
    # Merge assessment features
    # Adds cutoff-safe student results
    # ------------------------------------------------------------
    df = df.merge(
        assessment_features,
        on=KEYS,
        how="left"
    )

    # ------------------------------------------------------------
    # Fill missing engineered values
    # Missing means no activity/submission before cutoff
    # ------------------------------------------------------------
    engineered_cols = [
        col for col in df.columns
        if col not in demographic_cols
    ]

    df[engineered_cols] = df[engineered_cols].fillna(0)

    df["feature_set"] = f"week{cutoff_week}"

    return df

# Run Leakage check for forbidden future columns

In [28]:
def audit_no_leakage(df, cutoff_week):
    forbidden_columns = [
        "date",
        "date_submitted",
        "score",
        "id_assessment",
        "is_banked",
        "date_unregistration"
    ]

    found = [col for col in forbidden_columns if col in df.columns]

    if found:
        raise ValueError(
            f"Leakage risk in week {cutoff_week}: forbidden columns found: {found}"
        )

    if "target" not in df.columns:
        raise ValueError(f"Week {cutoff_week}: missing binary target column.")

    if df["target"].isna().sum() > 0:
        raise ValueError(f"Week {cutoff_week}: target contains missing values.")

    return True

# Build feature tables

In [29]:
def save_features_table_manifest(feature_tables):
    rows = []

    for week, df in feature_tables.items():
        for col in df.columns:

            if col in KEYS:
                source = "studentInfo"
                definition = "Student-course identifier."
                computation = "Retained from Stage 1 studentInfo."
                leakage = "None"

            elif col in [
                "gender",
                "region",
                "highest_education",
                "imd_band",
                "age_band",
                "num_of_prev_attempts",
                "studied_credits",
                "disability"
            ]:
                source = "studentInfo"
                definition = "Student demographic or registration feature."
                computation = "Retained from Stage 1 studentInfo."
                leakage = "Low"

            elif col in ["final_result", "target"]:
                source = "studentInfo"
                definition = "Outcome label."
                computation = "Created in Stage 1 from final_result."
                leakage = "High if used as input feature."

            elif col.startswith("clicks_"):
                source = "studentVle + vle"
                definition = "Clicks for a specific VLE activity type."
                computation = "Sum of sum_click by student and activity_type before cutoff."
                leakage = "None"

            elif col in [
                "total_clicks",
                "mean_daily_clicks",
                "max_daily_clicks",
                "active_days",
                "unique_sites",
                "unique_activity_types",
                "clicks_per_active_day",
                "clicks_per_site"
            ]:
                source = "studentVle"
                definition = "Aggregated VLE engagement before cutoff."
                computation = "Grouped aggregation using only rows up to cutoff week."
                leakage = "None"

            elif (
                col in [
                    "num_assessments_submitted",
                    "mean_assessment_score",
                    "max_assessment_score",
                    "min_assessment_score",
                    "total_weighted_score",
                    "mean_days_submitted",
                    "earliest_submission_day",
                    "latest_submission_day"
                ]
                or (col.startswith("num_") and col.endswith("_submitted"))
                or (col.startswith("mean_") and col.endswith("_score"))
            ):
                source = "studentAssessment + assessments"
                definition = "Aggregated assessment performance before cutoff."
                computation = "Grouped aggregation using only submissions before cutoff week."
                leakage = "None if filtered by cutoff"

            else:
                source = "engineered"
                definition = "Stage 3 helper or metadata feature."
                computation = "Created during feature engineering."
                leakage = "None"

            rows.append({
                "Week availability": week,
                "Feature": col,
                "Source CSV(s)": source,
                "Data type": str(df[col].dtype),
                "Definition": definition,
                "How computed": computation,
                "Missing count": int(df[col].isna().sum()),
                "Duplicate count": int(df.duplicated().sum()),
                "Leakage risk": leakage,
                "Notes": ""
            })

    manifest = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # Save feature documentation
    # Records source, meaning, and leakage risk
    # ------------------------------------------------------------
    out_path = os.path.join(PROCESSED_DIR, "stage3_features_table.csv")
    manifest.to_csv(out_path, index=False)

    print(f"Saved feature manifest: {out_path}")

# Run all week cutoffs

In [30]:
def main():
    student_info, student_vle, vle, assessments, student_assessment = load_stage3_inputs()

    student_vle_enriched = add_activity_type(student_vle, vle)

    assessment_enriched = add_assessment_metadata(
        student_assessment,
        assessments
    )

    feature_tables = {}

    for week in WEEK_CUTOFFS:
        print(f"\nBuilding week {week} feature table...")

        vle_features = aggregate_vle_until_cutoff(
            student_vle_enriched,
            cutoff_week=week
        )

        assessment_features = aggregate_assessments_until_cutoff(
            assessment_enriched,
            cutoff_week=week
        )

        week_df = build_week_feature_table(
            student_info,
            vle_features,
            assessment_features,
            cutoff_week=week
        )

        audit_no_leakage(week_df, cutoff_week=week)

        # ------------------------------------------------------------
        # Save week feature table
        # Output for Stages 4 and 5
        # ------------------------------------------------------------
        out_path = os.path.join(PROCESSED_DIR, f"week{week}_features.csv")
        week_df.to_csv(out_path, index=False)

        feature_tables[week] = week_df

        print(f"Saved: {out_path}")
        print(f"Shape: {week_df.shape[0]:,} rows x {week_df.shape[1]} columns")

        print("Target distribution:")
        print(week_df["target"].value_counts(normalize=True).round(3))

        assessment_cols = [
            col for col in week_df.columns
            if "assessment" in col or "score" in col or "submitted" in col
        ]

        print("Assessment feature preview:")
        print(week_df[assessment_cols].head())

    save_features_table_manifest(feature_tables)

    print("\nStage 3 complete.")
    print("Generated week2, week4, week6, and week8 feature tables with assessment features.")


if __name__ == "__main__":
    main()


Building week 2 feature table...
Saved: data/processed/week2_features.csv
Shape: 32,593 rows x 54 columns
Target distribution:
target
0    0.528
1    0.472
Name: proportion, dtype: float64
Assessment feature preview:
   num_assessments_submitted  mean_assessment_score  max_assessment_score  \
0                        0.0                    0.0                   0.0   
1                        0.0                    0.0                   0.0   
2                        0.0                    0.0                   0.0   
3                        0.0                    0.0                   0.0   
4                        0.0                    0.0                   0.0   

   min_assessment_score  total_weighted_score  mean_days_submitted  \
0                   0.0                   0.0                  0.0   
1                   0.0                   0.0                  0.0   
2                   0.0                   0.0                  0.0   
3                   0.0                

## Stage 4: Baseline Pipeline Construction

This section constructs and evaluates two baseline classifiers across the four week-cutoff feature tables produced in Stage 3. 

**Models Evaluated (Stratified 5-Fold CV):**
1.  `DummyClassifier(strategy='most_frequent')` — naive majority-class baseline.
2.  `LogisticRegression(max_iter=1000)` — primary linear baseline.

In [31]:
# Imports and Configuration
import os
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import OrdinalEncoder


warnings.filterwarnings("ignore", category=FutureWarning)

# Configuration
PROCESSED_DIR = os.path.join("data", "processed")
WEEK_CUTOFFS = [2, 4, 6, 8]
RANDOM_SEED = 42
N_FOLDS = 5

# Columns to drop before modelling
COLS_TO_DROP = [
    "final_result",        # Target leakage
    "id_student",          # Row identifier
    "code_module",         # Row identifier
    "code_presentation",   # Row identifier
    "cutoff_week",         # Metadata tag
    "cutoff_day",          # Metadata tag
    "feature_set",         # Metadata tag
]


# Scoring metrics for cross_validate (Updated to handle zero_division)
SCORING = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "auc": "roc_auc"
}

# Load the tables into memory variables to avoid repeated disk reads
feature_tables = {}
for week in WEEK_CUTOFFS:
    path = os.path.join(PROCESSED_DIR, f"week{week}_features.csv")
    feature_tables[week] = pd.read_csv(path)
    
print("Feature tables successfully loaded into memory variables.")

Feature tables successfully loaded into memory variables.


Build Preprocessors and pipelines

In [32]:
def load_and_clean(df: pd.DataFrame, week: int) -> tuple:
    # Separate target before dropping columns
    y = df["target"].copy()
    
    # Drop target + leaky / identifier / metadata columns
    drop_cols = ["target"] + [c for c in COLS_TO_DROP if c in df.columns]
    X = df.drop(columns=drop_cols)
    
    # Explicitly separate Ordinal and Nominal (Fixes Pandas str select_dtypes warning)
    ordinal_cols = ["highest_education", "imd_band", "age_band"]
    nominal_cols = [c for c in X.select_dtypes(include=["object", "category", "string"]).columns if c not in ordinal_cols]
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    
    print(f" Week {week}: {X.shape[0]:,} rows, \n"
          f" {len(nominal_cols)} nominal + {len(ordinal_cols)} ordinal + {len(num_cols)} numerical = "
          f"{X.shape[1]} features | target mean = {y.mean():.3f}")
          
    return X, y, nominal_cols, ordinal_cols, num_cols


def build_preprocessor(nominal_cols: list[str], ordinal_cols: list[str], num_cols: list[str]) -> ColumnTransformer:
    # Define exact category hierarchies
    edu_cats = ["No Formal quals", "Lower Than A Level", "A Level or Equivalent", "HE Qualification", "Post Graduate Qualification"]
    imd_cats = ["0-10%", "10-20%", "20-30%", "30-40%", "40-50%", "50-60%", "60-70%", "70-80%", "80-90%", "90-100%"]
    age_cats = ["0-35", "35-55", "55<="]
    
    nominal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    
    ordinal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[edu_cats, imd_cats, age_cats], handle_unknown="use_encoded_value", unknown_value=-1))
    ])
    
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    return ColumnTransformer(
        transformers=[
            ("nom", nominal_pipeline, nominal_cols),
            ("ord", ordinal_pipeline, ordinal_cols),
            ("num", numerical_pipeline, num_cols),
        ],
        remainder="drop"
    )

def build_pipelines(nominal_cols: list[str], ordinal_cols: list[str], num_cols: list[str]) -> dict[str, Pipeline]:
    return {
        "DUM": Pipeline([
            ("preprocessor", build_preprocessor(nominal_cols, ordinal_cols, num_cols)),
            ("classifier", DummyClassifier(strategy="most_frequent")),
        ]),
        "LR": Pipeline([
            ("preprocessor", build_preprocessor(nominal_cols, ordinal_cols, num_cols)),
            ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs")),
        ]),
    }




def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series) -> dict[str, float]:
    """
    Run Stratified K-Fold cross-validation and return mean ± std for each metric.
    """
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    cv_results = cross_validate(
        pipe, X, y,
        cv=cv,
        scoring=SCORING,
        return_train_score=False,
        n_jobs=-1,
    )

    summary = {}
    for metric_name in SCORING:
        key = f"test_{metric_name}"
        scores = cv_results[key]
        summary[f"{metric_name}_mean"] = scores.mean()
        summary[f"{metric_name}_std"]  = scores.std()

    return summary

Run experiments and create experiment table

In [33]:
# Experiment Execution
MODEL_FULL_NAMES = {
    "DUM": "DummyClassifier(most_frequent)",
    "LR":  "LogisticRegression(max_iter=1000)",
}

PREPROCESSING_DESC = (
    "ColumnTransformer: "
    "cat→SimpleImputer(most_frequent)+OneHotEncoder | "
    "num→SimpleImputer(median)+StandardScaler"
)

def run_all_experiments(tables_dict: dict) -> pd.DataFrame:
    """
    Loop over every (model, week) combination using the provided dictionary
    of DataFrames, evaluate with Stratified 5-Fold CV, and compile results.
    """
    rows = []
    run_counter = 0

    print("\n" + "=" * 72)
    print("  STAGE 4 — BASELINE PIPELINE EVALUATION")
    print("=" * 72)

    for week in WEEK_CUTOFFS:
        print(f"\n── Processing week {week} dataset ──")
        
        # Pull dataframe from dictionary variable
        df_week = tables_dict[week]
        X, y, nominal_cols, ordinal_cols, num_cols = load_and_clean(df_week, week)
        pipelines = build_pipelines(nominal_cols, ordinal_cols, num_cols)
 

        for model_tag, pipe in pipelines.items():
            run_id = f"R{run_counter}_{model_tag}_W{week}"
            print(f"\n  ▸ {run_id}  ({MODEL_FULL_NAMES[model_tag]})")

            metrics = evaluate_pipeline(pipe, X, y)

            row = {
                "Run ID":                   run_id,
                "Week cutoff":              week,
                "Feature set":              f"week{week}",
                "Preprocessing":            PREPROCESSING_DESC,
                "Model":                    MODEL_FULL_NAMES[model_tag],
                "Hyperparameters changed?": "None",
                "Split protocol":           f"Stratified {N_FOLDS}-Fold CV",
                "Seed":                     RANDOM_SEED,
                "Accuracy (mean)":          round(metrics["accuracy_mean"],  4),
                "Accuracy (std)":           round(metrics["accuracy_std"],   4),
                "Precision (mean)":         round(metrics["precision_mean"], 4),
                "Precision (std)":          round(metrics["precision_std"],  4),
                "Recall (mean)":            round(metrics["recall_mean"],    4),
                "Recall (std)":             round(metrics["recall_std"],     4),
                "F1 (mean)":                round(metrics["f1_mean"],        4),
                "F1 (std)":                 round(metrics["f1_std"],         4),
                "AUC (mean)":               round(metrics["auc_mean"],       4),
                "AUC (std)":                round(metrics["auc_std"],        4),
                "Notes":                    "Baseline — no hyperparameter tuning",
            }

            rows.append(row)
            run_counter += 1

            # Print a quick summary to console
            print(f"    Accuracy : {row['Accuracy (mean)']:.4f} ± {row['Accuracy (std)']:.4f}")
            print(f"    Precision: {row['Precision (mean)']:.4f} ± {row['Precision (std)']:.4f}")
            print(f"    Recall   : {row['Recall (mean)']:.4f} ± {row['Recall (std)']:.4f}")
            print(f"    F1       : {row['F1 (mean)']:.4f} ± {row['F1 (std)']:.4f}")
            print(f"    AUC      : {row['AUC (mean)']:.4f} ± {row['AUC (std)']:.4f}")

    return pd.DataFrame(rows)

# Execute the experiment loop using the loaded variables
experiments_df = run_all_experiments(feature_tables)

# Save to CSV
os.makedirs(PROCESSED_DIR, exist_ok=True)
out_path = os.path.join(PROCESSED_DIR, "stage4_experiments_baseline.csv")
experiments_df.to_csv(out_path, index=False)

print("\n" + "=" * 72)
print(f"  Experiments Table saved → {out_path}")
print(f"  {len(experiments_df)} runs recorded ({len(WEEK_CUTOFFS)} weeks × 2 models)")
print("=" * 72)

# Display Summary
display_cols = [
    "Run ID", "Week cutoff", "Model",
    "Accuracy (mean)", "Precision (mean)", "Recall (mean)",
    "F1 (mean)", "AUC (mean)",
]
display(experiments_df[display_cols])


  STAGE 4 — BASELINE PIPELINE EVALUATION

── Processing week 2 dataset ──
 Week 2: 32,593 rows, 
 3 nominal + 3 ordinal + 40 numerical = 46 features | target mean = 0.472

  ▸ R0_DUM_W2  (DummyClassifier(most_frequent))
    Accuracy : 0.5280 ± 0.0000
    Precision: 0.0000 ± 0.0000
    Recall   : 0.0000 ± 0.0000
    F1       : 0.0000 ± 0.0000
    AUC      : 0.5000 ± 0.0000

  ▸ R1_LR_W2  (LogisticRegression(max_iter=1000))
    Accuracy : 0.6954 ± 0.0043
    Precision: 0.6777 ± 0.0049
    Recall   : 0.6765 ± 0.0056
    F1       : 0.6771 ± 0.0045
    AUC      : 0.7686 ± 0.0047

── Processing week 4 dataset ──
 Week 4: 32,593 rows, 
 3 nominal + 3 ordinal + 40 numerical = 46 features | target mean = 0.472

  ▸ R2_DUM_W4  (DummyClassifier(most_frequent))
    Accuracy : 0.5280 ± 0.0000
    Precision: 0.0000 ± 0.0000
    Recall   : 0.0000 ± 0.0000
    F1       : 0.0000 ± 0.0000
    AUC      : 0.5000 ± 0.0000

  ▸ R3_LR_W4  (LogisticRegression(max_iter=1000))
    Accuracy : 0.7295 ± 0.0051
  

,Run ID,Week cutoff,Model,Accuracy (mean),Precision (mean),Recall (mean),F1 (mean),AUC (mean)
0,R0_DUM_W2,2,DummyClassifier(most_frequent),0.5280,0.0000,0.0000,0.0000,0.5000
1,R1_LR_W2,2,LogisticRegression(max_iter=1000),0.6954,0.6777,0.6765,0.6771,0.7686
2,R2_DUM_W4,4,DummyClassifier(most_frequent),0.5280,0.0000,0.0000,0.0000,0.5000
3,R3_LR_W4,4,LogisticRegression(max_iter=1000),0.7295,0.7107,0.7202,0.7154,0.8115
4,R4_DUM_W6,6,DummyClassifier(most_frequent),0.5280,0.0000,0.0000,0.0000,0.5000
5,R5_LR_W6,6,LogisticRegression(max_iter=1000),0.7423,0.7219,0.7387,0.7302,0.8265
6,R6_DUM_W8,8,DummyClassifier(most_frequent),0.5280,0.0000,0.0000,0.0000,0.5000
7,R7_LR_W8,8,LogisticRegression(max_iter=1000),0.7699,0.7405,0.7891,0.7640,0.8495


## Advanced Pipeline Construction

This section constructs and tunes advanced classifiers (Random Forest, XGBoost, and SGD/SVM) 
using RandomizedSearchCV across the four week-cutoff feature tables.

In [34]:
# if not installed run this for xgboost
# !pip install xgboost

In [36]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform


def build_advanced_pipelines(nominal_cols: list[str], ordinal_cols: list[str], num_cols: list[str]) -> tuple[dict, dict]:
    preprocessor = build_preprocessor(nominal_cols, ordinal_cols, num_cols)

    # Define Pipelines for all four models
    pipelines = {
        "RF": Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(random_state=RANDOM_SEED))
        ]),
        "XGB": Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'))
        ]),
        "SGD": Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", SGDClassifier(random_state=RANDOM_SEED, early_stopping=True))
        ]),
        "NN": Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", MLPClassifier(random_state=RANDOM_SEED, early_stopping=True, max_iter=500))
        ])
    }

    # Define Hyperparameter Grids for RandomizedSearchCV
    param_grids = {
        "RF": {
            "classifier__n_estimators": randint(100, 300),
            "classifier__max_depth": [None, 10, 20, 30],
            "classifier__min_samples_split": randint(2, 10),
            "classifier__class_weight": ["balanced", None]
        },
        "XGB": {
            "classifier__n_estimators": randint(100, 300),
            "classifier__max_depth": randint(3, 10),
            "classifier__learning_rate": uniform(0.01, 0.2),
            "classifier__scale_pos_weight": [1, 2, 3] 
        },
        "SGD": {
            "classifier__loss": ['hinge', 'log_loss', 'modified_huber'],
            "classifier__alpha": uniform(0.0001, 0.01),
            "classifier__penalty": ['l2', 'l1', 'elasticnet']
        },
        "NN": {
            "classifier__hidden_layer_sizes": [(50,), (100,), (50, 50)],
            "classifier__alpha": uniform(0.0001, 0.05),
            "classifier__learning_rate_init": uniform(0.001, 0.05)
        }
    }

    return pipelines, param_grids


def run_advanced_experiments(tables_dict: dict) -> pd.DataFrame:
    """
    Loop over every (model, week) combination, tune hyperparameters using RandomizedSearchCV 
    with Stratified 5-Fold CV, and compile results.
    """
    rows = []
    run_counter = 0

    print("\n" + "=" * 72)
    print("  STAGE 5 — ADVANCED PIPELINE TUNING")
    print("=" * 72)

    for week in WEEK_CUTOFFS:
        print(f"\n── Tuning week {week} dataset ──")
        
        df_week = tables_dict[week]
        X, y, nominal_cols, ordinal_cols, num_cols = load_and_clean(df_week, week)
        pipelines, param_grids = build_advanced_pipelines(nominal_cols, ordinal_cols, num_cols)
        
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

        for model_tag, pipe in pipelines.items():
            run_id = f"R{run_counter}_{model_tag}_W{week}_TUNED"
            print(f"\n  ▸ {run_id} tuning started...")

            # Set up RandomizedSearchCV
            # Refitting on AUC to prioritize robust probability ordering
            search = RandomizedSearchCV(
                pipe,
                param_distributions=param_grids[model_tag],
                n_iter=10,
                cv=cv,
                scoring=SCORING,
                refit="auc", 
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbose=1
            )

            # Fit the search
            search.fit(X, y)
            
            # Extract results for the best model found
            best_idx = search.best_index_
            cv_res = search.cv_results_
            
            # Formatting the hyperparameter dictionary to a clean string for the table
            best_params_str = ", ".join([f"{k.replace('classifier__', '')}: {v}" for k, v in search.best_params_.items()])

            row = {
                "Run ID":                   run_id,
                "Week cutoff":              week,
                "Feature set":              f"week{week}",
                "Preprocessing":            PREPROCESSING_DESC,
                "Model":                    f"{model_tag} (Tuned)",
                "Hyperparameters changed?": best_params_str,
                "Split protocol":           f"Stratified {N_FOLDS}-Fold CV",
                "Seed":                     RANDOM_SEED,
                "Accuracy (mean)":          round(cv_res["mean_test_accuracy"][best_idx],  4),
                "Accuracy (std)":           round(cv_res["std_test_accuracy"][best_idx],   4),
                "Precision (mean)":         round(cv_res["mean_test_precision"][best_idx], 4),
                "Precision (std)":          round(cv_res["std_test_precision"][best_idx],  4),
                "Recall (mean)":            round(cv_res["mean_test_recall"][best_idx],    4),
                "Recall (std)":             round(cv_res["std_test_recall"][best_idx],     4),
                "F1 (mean)":                round(cv_res["mean_test_f1"][best_idx],        4),
                "F1 (std)":                 round(cv_res["std_test_f1"][best_idx],         4),
                "AUC (mean)":               round(cv_res["mean_test_auc"][best_idx],       4),
                "AUC (std)":                round(cv_res["std_test_auc"][best_idx],        4),
                "Notes":                    "Best estimator from RandomizedSearchCV",
            }

            rows.append(row)
            run_counter += 1

            print(f"    Best Params: {best_params_str}")
            print(f"    AUC (mean) : {row['AUC (mean)']:.4f} ± {row['AUC (std)']:.4f}")
            print(f"    F1 (mean)  : {row['F1 (mean)']:.4f} ± {row['F1 (std)']:.4f}")

    return pd.DataFrame(rows)


advanced_experiments_df = run_advanced_experiments(feature_tables)

# Save to CSV
out_path_adv = os.path.join(PROCESSED_DIR, "stage5_experiments_advanced.csv")
advanced_experiments_df.to_csv(out_path_adv, index=False)

print("\n" + "=" * 72)
print(f"  Advanced Experiments Table saved → {out_path_adv}")
print(f"  {len(advanced_experiments_df)} runs recorded ({len(WEEK_CUTOFFS)} weeks × 3 tuned models)")
print("=" * 72)


  STAGE 5 — ADVANCED PIPELINE TUNING

── Tuning week 2 dataset ──
 Week 2: 32,593 rows, 
 3 nominal + 3 ordinal + 40 numerical = 46 features | target mean = 0.472

  ▸ R0_RF_W2_TUNED tuning started...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
    Best Params: class_weight: balanced, max_depth: 20, min_samples_split: 4, n_estimators: 269
    AUC (mean) : 0.7793 ± 0.0042
    F1 (mean)  : 0.7037 ± 0.0048

  ▸ R1_XGB_W2_TUNED tuning started...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
    Best Params: learning_rate: 0.041198904067240534, max_depth: 5, n_estimators: 187, scale_pos_weight: 1
    AUC (mean) : 0.7852 ± 0.0041
    F1 (mean)  : 0.7032 ± 0.0044

  ▸ R2_SGD_W2_TUNED tuning started...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
    Best Params: alpha: 0.004419450186421158, loss: hinge, penalty: elasticnet
    AUC (mean) : 0.7678 ± 0.0040
    F1 (mean)  : 0.6865 ± 0.0173

  ▸ R3_NN_W2_TUNED tuning started...
Fitting 5 folds for

In [37]:
# Combine Baseline and Advanced Results to generate the final Pipeline Comparison Table
all_experiments_df = pd.concat([experiments_df, advanced_experiments_df], ignore_index=True)
combined_out_path = os.path.join(PROCESSED_DIR, "all_experiments_combined.csv")
all_experiments_df.to_csv(combined_out_path, index=False)

# Display Summary
display_cols = [
    "Run ID", "Week cutoff", "Model",
    "F1 (mean)", "AUC (mean)", "Hyperparameters changed?"
]
display(all_experiments_df[display_cols].sort_values(by=["Week cutoff", "AUC (mean)"], ascending=[True, False]))

,Run ID,Week cutoff,Model,F1 (mean),AUC (mean),Hyperparameters changed?
9,R1_XGB_W2_TUNED,2,XGB (Tuned),0.7032,0.7852,"learning_rate: 0.041198904067240534, max_depth..."
8,R0_RF_W2_TUNED,2,RF (Tuned),0.7037,0.7793,"class_weight: balanced, max_depth: 20, min_sam..."
11,R3_NN_W2_TUNED,2,NN (Tuned),0.6998,0.7787,"alpha: 0.03908455001363847, hidden_layer_sizes..."
1,R1_LR_W2,2,LogisticRegression(max_iter=1000),0.6771,0.7686,None
10,R2_SGD_W2_TUNED,2,SGD (Tuned),0.6865,0.7678,"alpha: 0.004419450186421158, loss: hinge, pena..."
0,R0_DUM_W2,2,DummyClassifier(most_frequent),0.0000,0.5000,None
13,R5_XGB_W4_TUNED,4,XGB (Tuned),0.7577,0.8412,"learning_rate: 0.041198904067240534, max_depth..."
12,R4_RF_W4_TUNED,4,RF (Tuned),0.7564,0.8367,"class_weight: balanced, max_depth: 20, min_sam..."
15,R7_NN_W4_TUNED,4,NN (Tuned),0.7474,0.8342,"alpha: 0.016785430556951093, hidden_layer_size..."
3,R3_LR_W4,4,LogisticRegression(max_iter=1000),0.7154,0.8115,None
